In [ ]:
from pathlib import Path

import h5py
import numpy as np
import torch
from torch.utils.data import DataLoader, Subset

import fluidbench
from fluidbench.cfdataset import CFDataset

In [ ]:
ID = "cylinder_Re41_interp"
dataset_path = fluidbench.download(ID)

In [ ]:
def load_split(split):
    folder = dataset_path / split
    cases = sorted((path for path in folder.glob("*.h5") if path.stem != "constants"), key=lambda path: int(path.stem))
    data = []
    for case in cases:
        with h5py.File(case, "r") as file:
            data.append((file["scalars"][:], file["fields"][:]))
    return data

def to_dataset(cases):
    # Each snapshot is paired with the next one: inputs are snapshot i, targets snapshot i + 1.
    scalars = np.concatenate([scalars[:-1] for scalars, fields in cases])
    inputs = np.concatenate([fields[:-1] for scalars, fields in cases])
    targets = np.concatenate([fields[1:] for scalars, fields in cases])
    return CFDataset(scalars, inputs, targets, np.arange(len(targets)))

In [ ]:
# The benchmark ships train and test only. Hold out a random 10% of the training pairs for validation,
# so validation covers the whole trajectory (early transient and settled flow) just like training does.
VAL_FRACTION = 0.05
train_val_dataset = to_dataset(load_split("train"))
indices = np.random.default_rng(0).permutation(len(train_val_dataset))
n_validation = int(len(indices) * VAL_FRACTION)
validation_dataset = Subset(train_val_dataset, indices[:n_validation])
train_dataset = Subset(train_val_dataset, indices[n_validation:])

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
validation_loader = DataLoader(validation_dataset, batch_size=8, shuffle=False)
len(train_dataset), len(validation_dataset)

In [ ]:
### Model definition
from models.unet import UNetClassic as architecture

weights_path = Path("weights") / ID / f"{architecture.__name__}.pt"

model = architecture()
model.to(model.device)
sum(p.numel() for p in model.parameters())

In [ ]:
history = model.fit(train_loader, validation_loader, epochs=1200)

In [ ]:
weights_path.parent.mkdir(parents=True, exist_ok=True)
model.save_weights(weights_path)
weights_path

In [ ]:
# Metrics are calculated by fluidbench.run() from the benchmark test split.
submission_config = {
    "datasetId": ID,
    "id": "unet-classic",
    "model": "UNet Classic",
    "authors": "Pablo A.C.",
    "year": 2026,
    "representation": "Grid",
}

# Optional: training curves from the fit above
training_history = {
    "train_error_history": history["train_losses"],
    "val_error_history": history["val_losses"],
    "train_error_name": history["loss_name"],
    "val_split": VAL_FRACTION,
}

In [ ]:
submission = fluidbench.run(model, submission_config, training_history)